In [10]:
import os
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "qwen/qwen3.8-27b",
    model_provider="groq",
    temperature=0,
    max_tokens=512
    
)

In [12]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The year the movie was released")
    director: str = Field(description="The director of the movie")
    rating:float =Field(description="The movies rating out of 10")
    Actor_Actress:str =Field(description="ead Actor or Actress ")

In [13]:
model_wstr= model.with_structured_output(Movie)

In [14]:
model_wstr.invoke("provide the detail about ")

Movie(title='Fight Club', year=1999, director='David Fincher', rating=8.2, Actor_Actress='Brad Pitt')

## Message Output alongside parsed structure 

In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""

    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie rating out of 10")
    Actor_Actress: str =Field(...,description=list"Lead actor and actress")


model_wstr= model.with_structured_output(Movie)

response = model_wstr.invoke("Provide details about the movie the inception ")

response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8, Actor_Actress='Leonardo DiCaprio, Joseph Gordon-Levitt, Elliot Page, Tom Hardy, Ken Watanabe, Marion Cotillard')

## Nested Structure 

In [20]:
from pydantic import BaseModel ,Field

class Actor (BaseModel):
    name:str
    role:str 

class MovieDetails(BaseModel):
    title:str 
    year:int
    cast: list[Actor]
    genres: list[str]
    budget:float | None =Field(None,description="Budget in millions USD")

In [23]:
model_wstr=model.with_structured_output(MovieDetails)
response = model_wstr.invoke("provide details about the movie inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Marion Cotillard', role='Mal Cobb'), Actor(name='Cillian Murphy', role='Robert Fischer')], genres=['Action', 'Science Fiction', 'Thriller'], budget=160.0)

# Type Dict 
####  TypeDict provider a simpler alternative using Python built_in typing ,ideal when you do not need runtime validation 

In [27]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details"""

    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie rating out of 10"]


model_with_typedict = model.with_structured_output(MovieDict)

response = model_with_typedict.invoke(
    "Please provide the details of the movie Avengers"
)

print(response)

{'director': 'Joss Whedon', 'rating': 8, 'title': 'Avengers', 'year': 2012}


In [28]:
 
class Actor(TypedDict) :
    name:str
    role:str
class MovieDetails(TypedDict):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget: float | None =Field(None, description="Budget in millions USD")   

model_with_typedict=model.with_structured_output(MovieDetails)   


response =model_with_typedict.invoke("Provide details about the movie Avengers")
response 




{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / The Hulk'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'},
  {'name': 'Tom Hiddleston', 'role': 'Loki'},
  {'name': 'Paul Bettany', 'role': 'J.A.R.V.I.S.'}],
 'genres': ['Action', 'Adventure', 'Science Fiction'],
 'title': 'Avengers',
 'year': 2012}

In [29]:
model.profile 


# DataClasses 
####  A data class is a class typically containing mainly data ,although there arenot really any restrictions .you create it using the @dataclass decorator 

In [31]:
import os 
from dotenv import load_dotenv
load_dotenv()

key=os.getenv("GROQ_API_KEY")


In [ ]:
key

'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'

In [35]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model


class ContactInfo(BaseModel):
    """Contact information for a person."""

    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")


model = init_chat_model(
    "openai/gpt-oss-20b",
    model_provider="groq"
)


agent = create_agent(
    model=model,
    response_format=ContactInfo
)


result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Extract contact info: John Doe, john@example.com, (555) 123-4567"
            }
        ]
    }
)


print(result)

{'messages': [HumanMessage(content='Extract contact info: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='6678d4b9-7587-4f3c-88c4-a2cc249b551c'), AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={'reasoning_content': 'We need to output JSON object conforming to schema ContactInfo. Fields: name, email, phone. Must be compact JSON. Ensure all fields present. Output only the JSON object. Let\'s produce: {"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}'}, response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 237, 'total_tokens': 337, 'completion_time': 0.108427038, 'completion_tokens_details': {'reasoning_tokens': 64}, 'prompt_time': 0.020926605, 'prompt_tokens_details': None, 'queue_time': 0.358439713, 'total_time': 0.129353643}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_d3e146e1a5', 'service_tier': 'on_demand', 'fini

In [36]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [38]:
from typing_extensions import TypedDict
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model


class ContactInfo(TypedDict):
    """Contact information for a person."""

    name: str       # The name of the person
    email: str      # The email address of the person
    phone: str      # The phone number of the person


model = init_chat_model(
    "openai/gpt-oss-20b",
    model_provider="groq"
)


agent = create_agent(
    model=model,
    response_format=ContactInfo
)


result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Extract contact info from: John Doe, john@example.com, (555)123-4567"
            }
        ]
    }
)


print(result["structured_response"])

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555)123-4567'}


In [39]:
# Dataclass 
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass 
class ContactInfo:
    """ Contact information for a person ."""
    name:str # the name of the person 
    email:str
    phone:str

agent =create_agent(  
    model = init_chat_model(
     "openai/gpt-oss-20b",
      model_provider="groq"
)

)    
result=agent.invoke({
    "messages":[{"role":"user","content":"Extract contact info from :John Doe , john@example.com ,(555)123-4567"}]


})

result ["structured_response"]

KeyError: 'structured_response'